In [1]:
# Célula 1 — instalar
!pip install jupysql pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 9.7 MB/s eta 0:00:00


In [2]:
# importing all dependencies


import pandas as pd
import sqlite3
from google.colab import drive
import os

In [3]:
# loading google colab and creating save point on google drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/MozDev_AI_Automation_Scripts/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive


In [15]:
ls "drive/"

ls: cannot access 'drive/': No such file or directory


In [16]:
cd "drive/MyDrive/MozDev_AI_Automation_Scripts/"

[Errno 2] No such file or directory: 'drive/MyDrive/MozDev_AI_Automation_Scripts/'
/content/drive/MyDrive/MozDev_AI_Automation_Scripts


In [22]:
# cirando uma database
conn = sqlite3.connect("mozchapa100.db")
pd.read_excel("MozChapa100_Segmentos_ERROS.xlsx", header=1).to_sql("segmentos", conn, if_exists="replace", index=False)
pd.read_excel("MozChapa100_Vendas_ERROS.xlsx",    header=1).to_sql("vendas",    conn, if_exists="replace", index=False)
pd.read_excel("MozChapa100_Rotas_ERROS.xlsx",     header=1).to_sql("rotas",     conn, if_exists="replace", index=False)
print("✅ DB criada!")

✅ DB criada!


In [23]:
# Cctivar SQL magic
%load_ext sql
%sql sqlite:///mozchapa100.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [25]:
%%sql
CREATE TABLE MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS AS
SELECT
    v.Data AS dia,
    s.Tipo_Passageiro AS segmento,
    (r.Origem || ' - ' || r.Destino) AS rota,
    v.Forma_Pagamento,

    SUM(v.Bilhetes_Vendidos) AS Bilhetes_Vendidos,
    AVG(v.Preco_Unitario_MZN) AS Preco_Unitario_MZN,

    SUM(r.Viagens_Realizadas) AS viagens,
    SUM(r.Passageiros_Total) AS passageiros,
    AVG(r.Tempo_Medio_Min) AS duracao_min,

    SUM(v.Bilhetes_Vendidos * v.Preco_Unitario_MZN) AS receita

FROM Vendas v
JOIN Segmentos s
    ON v.ID_Segmento = s.ID_Segmento
JOIN Rotas r
    ON v.ID_Rota = r.ID_Rota
   AND v.Data = r.Data

GROUP BY
    v.Data,
    s.Tipo_Passageiro,
    (r.Origem || ' - ' || r.Destino),
    v.Forma_Pagamento;

Running query in 'sqlite:///mozchapa100.db'

++
||
++
++

In [26]:
%%sql
SELECT * FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,viagens,passageiros,duracao_min,receita
2026-03-01 00:00:00,Estudante,Maputo Centro - Matola,M-Pesa,460.0,25.0,1380.0,64308.0,40.0,11500.0
2026-03-01 00:00:00,Trabalhador,Matola - Boane,e-Mola,1472.0,75.0,1472.0,59616.0,74.0,110400.0
2026-03-01 00:00:00,Turista,Machava - Marracuene,e-Mola,-460.0,75.0,828.0,19872.0,71.0,-34500.0
2026-03-01 00:00:00,Turista,Maputo Centro - KaMavota,Numerário,1840.0,100.0,1104.0,29164.0,21.0,184000.0
2026-03-01 00:00:00,Turista,Maputo Centro - Matola,M-Pesa,1564.0,75.0,1380.0,64308.0,40.0,117300.0
2026-03-02 00:00:00,Estudante,Machava - Marracuene,Numerário,1288.0,50.0,460.0,4876.0,75.0,64400.0
2026-03-02 00:00:00,Estudante,Maputo Centro - KaMavota,M-Pesa,736.0,50.0,1288.0,30176.0,32.0,36800.0
2026-03-02 00:00:00,Estudante,Matola - Boane,Numerário,92.0,50.0,1288.0,23644.0,55.0,4600.0
2026-03-02 00:00:00,Trabalhador,Maputo Centro - KaMavota,e-Mola,736.0,50.0,1288.0,30176.0,32.0,36800.0
2026-03-02 00:00:00,Turista,Machava - Marracuene,M-Pesa,1288.0,75.0,460.0,4876.0,75.0,96600.0


# Cleaning and fun part

## ✅ 1. NEGATIVO — numeric values < 0

In [27]:
%%sql
SELECT *
FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS
WHERE Bilhetes_Vendidos < 0
   OR Preco_Unitario_MZN < 0
   OR viagens < 0
   OR passageiros < 0
   OR duracao_min < 0
   OR receita < 0;

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,viagens,passageiros,duracao_min,receita
2026-03-01 00:00:00,Turista,Machava - Marracuene,e-Mola,-460.0,75.0,828.0,19872.0,71.0,-34500.0
2026-03-05 00:00:00,Turista,Maputo Centro - KaMavota,e-Mola,1380.0,-35.0,1104.0,32476.0,21.0,-48300.0
2026-03-11 00:00:00,Trabalhador,Machava - Marracuene,Numerário,-644.0,50.0,920.0,13432.0,82.0,-32200.0
2026-03-17 00:00:00,Turista,Maputo Centro - Matola,e-Mola,736.0,-75.0,1196.0,7820.0,57.0,-55200.0
2026-03-24 00:00:00,Turista,Maputo Centro - Machava,e-Mola,-1196.0,100.0,920.0,3772.0,32.0,-119600.0
2026-04-04 00:00:00,Trabalhador,Machava - Marracuene,M-Pesa,552.0,-100.0,736.0,12420.0,77.0,-55200.0


✅ 2. ESPAÇOS — dirty payment values

In [28]:
%%sql
SELECT *
FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS
WHERE Forma_Pagamento <> TRIM(Forma_Pagamento);

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,viagens,passageiros,duracao_min,receita
2026-03-08 00:00:00,Estudante,Maputo Centro - Machava,e-Mola,368.0,35.0,1472.0,67252.0,35.0,12880.0
2026-03-21 00:00:00,Turista,Machava - Marracuene,e-Mola,828.0,100.0,828.0,20976.0,85.0,82800.0
2026-03-31 00:00:00,Estudante,Matola - Boane,M-Pesa,184.0,35.0,1288.0,51520.0,64.0,6440.0


In [29]:
%%sql
SELECT *
FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS
WHERE Forma_Pagamento LIKE '%  %';

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,viagens,passageiros,duracao_min,receita


✅ 3. DATA_ERRADA — datetime instead of pure date

In [30]:
%%sql
SELECT *
FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS
WHERE CAST(dia AS DATE) <> dia;

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,viagens,passageiros,duracao_min,receita
2026-03-01 00:00:00,Estudante,Maputo Centro - Matola,M-Pesa,460.0,25.0,1380.0,64308.0,40.0,11500.0
2026-03-01 00:00:00,Trabalhador,Matola - Boane,e-Mola,1472.0,75.0,1472.0,59616.0,74.0,110400.0
2026-03-01 00:00:00,Turista,Machava - Marracuene,e-Mola,-460.0,75.0,828.0,19872.0,71.0,-34500.0
2026-03-01 00:00:00,Turista,Maputo Centro - KaMavota,Numerário,1840.0,100.0,1104.0,29164.0,21.0,184000.0
2026-03-01 00:00:00,Turista,Maputo Centro - Matola,M-Pesa,1564.0,75.0,1380.0,64308.0,40.0,117300.0
2026-03-02 00:00:00,Estudante,Machava - Marracuene,Numerário,1288.0,50.0,460.0,4876.0,75.0,64400.0
2026-03-02 00:00:00,Estudante,Maputo Centro - KaMavota,M-Pesa,736.0,50.0,1288.0,30176.0,32.0,36800.0
2026-03-02 00:00:00,Estudante,Matola - Boane,Numerário,92.0,50.0,1288.0,23644.0,55.0,4600.0
2026-03-02 00:00:00,Trabalhador,Maputo Centro - KaMavota,e-Mola,736.0,50.0,1288.0,30176.0,32.0,36800.0
2026-03-02 00:00:00,Turista,Machava - Marracuene,M-Pesa,1288.0,75.0,460.0,4876.0,75.0,96600.0


✅ Cleaning Done

In [31]:
%%sql
CREATE TABLE MOZ_DEV_MOZCHAPPA100_CLEAN AS
SELECT
    DATE(dia) AS dia,   -- removes time if datetime exists

    segmento,

    rota,

    TRIM(Forma_Pagamento) AS Forma_Pagamento,

    ABS(Bilhetes_Vendidos) AS Bilhetes_Vendidos,
    ABS(Preco_Unitario_MZN) AS Preco_Unitario_MZN,

    ABS(viagens) AS viagens,
    ABS(passageiros) AS passageiros,
    ABS(duracao_min) AS duracao_min,

    ABS(receita) AS receita

FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS;

Running query in 'sqlite:///mozchapa100.db'

++
||
++
++

In [32]:
%%sql
SELECT * FROM MOZ_DEV_MOZCHAPPA100_CLEAN

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,viagens,passageiros,duracao_min,receita
2026-03-01,Estudante,Maputo Centro - Matola,M-Pesa,460.0,25.0,1380.0,64308.0,40.0,11500.0
2026-03-01,Trabalhador,Matola - Boane,e-Mola,1472.0,75.0,1472.0,59616.0,74.0,110400.0
2026-03-01,Turista,Machava - Marracuene,e-Mola,460.0,75.0,828.0,19872.0,71.0,34500.0
2026-03-01,Turista,Maputo Centro - KaMavota,Numerário,1840.0,100.0,1104.0,29164.0,21.0,184000.0
2026-03-01,Turista,Maputo Centro - Matola,M-Pesa,1564.0,75.0,1380.0,64308.0,40.0,117300.0
2026-03-02,Estudante,Machava - Marracuene,Numerário,1288.0,50.0,460.0,4876.0,75.0,64400.0
2026-03-02,Estudante,Maputo Centro - KaMavota,M-Pesa,736.0,50.0,1288.0,30176.0,32.0,36800.0
2026-03-02,Estudante,Matola - Boane,Numerário,92.0,50.0,1288.0,23644.0,55.0,4600.0
2026-03-02,Trabalhador,Maputo Centro - KaMavota,e-Mola,736.0,50.0,1288.0,30176.0,32.0,36800.0
2026-03-02,Turista,Machava - Marracuene,M-Pesa,1288.0,75.0,460.0,4876.0,75.0,96600.0


In [33]:
import pandas as pd

df = %sql SELECT * FROM MOZ_DEV_MOZCHAPPA100_CLEAN
df = df.DataFrame()

df.to_excel("MozChapa100_CLEAN.xlsx", index=False)

Running query in 'sqlite:///mozchapa100.db'